In [2]:
! pip install nba_api


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.endpoints import boxscoretraditionalv3
import os

os.makedirs("data/raw", exist_ok=True)


from pathlib import Path

file_path = Path("data/raw/team_games.csv")

print(file_path.exists())



def fetch_team_games():
    games = leaguegamefinder.LeagueGameFinder().get_data_frames()[0] ## using the league game finder part of the api to find games and the zero is for pulling the first dataframe which is the one we need
    games.to_csv ("data/raw/team_games.csv", index= False)
    return games


def fetch_player_games(team_games_df):
    all_players = []
    game_ids = team_games_df["GAME_ID"].unique()

    for game_id in game_ids[:200]:
      try:
          box =boxscoretraditionalv3.BoxScoreTraditionalV3(game_id = game_id)
          players = box.get_data_frames()[0]
          all_players.append(players)
          time.sleep(0.6 ) ## to avoid exceeding the rate limit
      except:
          continue

    players_df = pd.concat(all_players)
    players_df.to_csv("data/raw/player_games.csv", index=False)
    return players_df


True


In [ ]:
team_games = fetch_team_games()####### WE USE THIS FUNCTION TO FETCH THE TEAM GAMES AND SAVE IT TO A CSV FOR FUTURE USE>IT ALSO RETURNS THE DATA FRAME THAT WE CAN USE TO FETCH THE PLAYER GAMES
team_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612744,GSW,Golden State Warriors,0022500820,2026-02-22,GSW vs. DEN,W,241,128,...,0.917,9,35,44,42,14,2,15,24,11.0
1,22025,1610612739,CLE,Cleveland Cavaliers,0022500817,2026-02-22,CLE @ OKC,L,241,113,...,0.737,12,32,44,28,9,3,17,22,-8.0
2,22025,1610612741,CHI,Chicago Bulls,0022500825,2026-02-22,CHI vs. NYK,L,240,99,...,0.714,13,37,50,24,8,2,17,17,-6.0
3,22025,1610612751,BKN,Brooklyn Nets,0022500818,2026-02-22,BKN @ ATL,L,240,104,...,0.727,9,25,34,29,9,4,13,18,-11.0
4,22025,1610612754,IND,Indiana Pacers,0022500821,2026-02-22,IND vs. DAL,L,241,130,...,0.800,9,30,39,37,8,3,15,23,-4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,22014,1610612752,NYK,New York Knicks,0021400950,2015-03-10,NYK @ UTA,L,239,82,...,0.882,8,30,38,17,5,5,12,22,-5.0
29996,22014,1610612753,ORL,Orlando Magic,0021400946,2015-03-10,ORL @ IND,L,241,86,...,0.700,19,26,45,15,9,7,10,17,-32.0
29997,22014,1610612761,TOR,Toronto Raptors,0021400949,2015-03-10,TOR @ SAS,L,240,107,...,0.722,18,28,46,19,6,5,9,26,-10.0
29998,22014,1610612748,MIA,Miami Heat,0021400940,2015-03-09,MIA vs. BOS,L,239,90,...,0.824,5,27,32,15,5,2,13,17,-10.0


In [ ]:
player_stats= fetch_player_games(team_games)## team_games is important since it parses the the game stats and gives use the the game-ids theat

In [ ]:
os.makedirs("data/processsed", exist_ok=True)

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/team_games.csv")
print(df.head())
print(df.columns)



df['GAME_DATE']=pd.to_datetime(df['GAME_DATE'])
df = df.sort_values('GAME_DATE')
df = df.drop_duplicates()

df['WIN'] = df['WL'].apply(lambda x: 1 if x == 'W' else 0)


team_columns = [
    'GAME_DATE','TEAM_ID','TEAM_NAME','GAME_ID',
    'PTS','REB','AST','TOV',
    'FGM','FGA','FG_PCT',
    'FG3M','FG3A','FG3_PCT',
    'FTM','FTA','FT_PCT',
    'WIN'
]

df =df[team_columns]

df.to_csv("data/processsed/team_games.csv", index=False)



   SEASON_ID     TEAM_ID TEAM_ABBREVIATION              TEAM_NAME   GAME_ID  \
0      22025  1610612744               GSW  Golden State Warriors  22500820   
1      22025  1610612739               CLE    Cleveland Cavaliers  22500817   
2      22025  1610612741               CHI          Chicago Bulls  22500825   
3      22025  1610612751               BKN          Brooklyn Nets  22500818   
4      22025  1610612754               IND         Indiana Pacers  22500821   

    GAME_DATE      MATCHUP WL  MIN  PTS  ...  FT_PCT  OREB  DREB  REB  AST  \
0  2026-02-22  GSW vs. DEN  W  241  128  ...   0.917     9    35   44   42   
1  2026-02-22    CLE @ OKC  L  241  113  ...   0.737    12    32   44   28   
2  2026-02-22  CHI vs. NYK  L  240   99  ...   0.714    13    37   50   24   
3  2026-02-22    BKN @ ATL  L  240  104  ...   0.727     9    25   34   29   
4  2026-02-22  IND vs. DAL  L  241  130  ...   0.800     9    30   39   37   

   STL  BLK  TOV  PF  PLUS_MINUS  
0   14    2   15  24 

In [2]:
stats_rolling =[ 
    'PTS','REB','AST','TOV',
    'FGM','FGA','FG_PCT',
    'FG3M','FG3A','FG3_PCT',
    'FTM','FTA','FT_PCT',
    ]

df= df.sort_values(['TEAM_ID', 'GAME_DATE', 'GAME_ID'])


for stat in stats_rolling:
    df[f"rolling_{stat}"]= df.groupby('TEAM_ID')[stat].shift(1).rolling(5).mean()

In [3]:
[col for col in df.columns if col.startswith("rolling_")]

['rolling_PTS',
 'rolling_REB',
 'rolling_AST',
 'rolling_TOV',
 'rolling_FGM',
 'rolling_FGA',
 'rolling_FG_PCT',
 'rolling_FG3M',
 'rolling_FG3A',
 'rolling_FG3_PCT',
 'rolling_FTM',
 'rolling_FTA',
 'rolling_FT_PCT']

In [4]:
df[[col for col in df.columns if col.startswith("rolling_")]].isna().sum() ## to check how many missing values we have in the rolling stats columns

rolling_PTS        285
rolling_REB        285
rolling_AST        285
rolling_TOV        285
rolling_FGM        285
rolling_FGA        285
rolling_FG_PCT     295
rolling_FG3M       285
rolling_FG3A       285
rolling_FG3_PCT    297
rolling_FTM        285
rolling_FTA        285
rolling_FT_PCT     303
dtype: int64

In [5]:
df.groupby('TEAM_ID')["rolling_PTS"].apply(lambda x: x.isna().sum())

TEAM_ID
93            5
94            1
104           2
12304         1
12315         3
             ..
1610616861    2
1610616862    2
1610616863    2
1610616864    1
1610616865    1
Name: rolling_PTS, Length: 78, dtype: int64

In [6]:
opp_df = df.copy()# we create a copy of the original data frame to create the opponent stats data frame
rolling_cols = [cols for cols in df.columns if cols.startswith("rolling_")]# we create a list of the rolling stats columns to rename them with the prefix "opp_" to indicate that they are the opponent stats
opp_df = opp_df.rename(columns={cols: f"opp_{cols}" for cols in rolling_cols}) # we rename the columns of the opponent data frame with the prefix "opp_" to indicate that they are the opponent stats


df = df.merge(opp_df,
              on = 'GAME_ID',
              suffixes=("", "_drop"))

In [7]:
print(df.columns)

Index(['GAME_DATE', 'TEAM_ID', 'TEAM_NAME', 'GAME_ID', 'PTS', 'REB', 'AST',
       'TOV', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA',
       'FT_PCT', 'WIN', 'rolling_PTS', 'rolling_REB', 'rolling_AST',
       'rolling_TOV', 'rolling_FGM', 'rolling_FGA', 'rolling_FG_PCT',
       'rolling_FG3M', 'rolling_FG3A', 'rolling_FG3_PCT', 'rolling_FTM',
       'rolling_FTA', 'rolling_FT_PCT', 'GAME_DATE_drop', 'TEAM_ID_drop',
       'TEAM_NAME_drop', 'PTS_drop', 'REB_drop', 'AST_drop', 'TOV_drop',
       'FGM_drop', 'FGA_drop', 'FG_PCT_drop', 'FG3M_drop', 'FG3A_drop',
       'FG3_PCT_drop', 'FTM_drop', 'FTA_drop', 'FT_PCT_drop', 'WIN_drop',
       'opp_rolling_PTS', 'opp_rolling_REB', 'opp_rolling_AST',
       'opp_rolling_TOV', 'opp_rolling_FGM', 'opp_rolling_FGA',
       'opp_rolling_FG_PCT', 'opp_rolling_FG3M', 'opp_rolling_FG3A',
       'opp_rolling_FG3_PCT', 'opp_rolling_FTM', 'opp_rolling_FTA',
       'opp_rolling_FT_PCT'],
      dtype='object')


In [8]:
df = df[df["TEAM_ID"] != df["TEAM_ID_drop"]]

In [9]:
df =df.drop(columns=[cols for cols in df.columns if cols.endswith("_drop")])

In [ ]:
print(df.columns)

In [11]:

feature_cols =[col for col in df.columns if "rolling" in col]
df = df.dropna(subset=feature_cols).reset_index(drop=True)

In [ ]:
print(df.columns)

In [13]:
df.to_csv("data/processsed/team_games_with_rolling_stats.csv", index=False)

import json
with open("data/processsed/feature_cols.json", "w") as f:
    json.dump(feature_cols, f)